Purpose: Identify targets for motif enrichment (genes that are only cis-regulated/HEB, not trans-regulated/parental DE).<br>
Author: Anna Pardo<br>
Date initiated: Feb. 25, 2026

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json

In [2]:
# load CAM gene annotation - will be needed later
cam = pd.read_csv(os.path.join("/home/leviathan22/Yucca_genomics/rna_insilico_genome/degs_downstream/",
                              "camgenes_Ya_Yf_orthology_synteny.txt"),sep="\t",header="infer")
cam.head()

,GeneID,Orthogroup,Pathway,gene_name,gene_abbr,gene_abbr_unique,subgenome
0,Yucal.01G165600.v2.1,OG0001578,CAM-dark,beta-carbonic anhydrase,bCA1234,Ya_bCA1234_1,Ya
1,Yucal.02G112700.v2.1,OG0001578,CAM-dark,beta-carbonic anhydrase,bCA1234,Ya_bCA1234_2,Ya
2,Yucal.04G001000.v2.1,OG0004406,CAM-dark,beta-carbonic anhydrase,bCA5,Ya_bCA5_1,Ya
3,Yucal.07G000800.v2.1,OG0004406,CAM-dark,beta-carbonic anhydrase,bCA5,Ya_bCA5_2,Ya
4,Yucal.03G120900.v2.1,OG0002899,CAM-dark,NAD-dependent malate dehydrogenase (chloroplas...,NAD-MDH-cp,Ya_NAD-MDH-cp_1,Ya


In [3]:
# load HEB results
heb = pd.read_csv(os.path.join("/home/leviathan22/Yucca_genomics/rna_insilico_genome/heb_deseq_outfiles/",
                               "deseq_HEB_with_additional_cols.txt"),sep="\t",header="infer")
heb.head()

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,syntelogID,Contrast,Yf_GeneID,Ya_GeneID,genotype,subgenome_bias
0,67.211469,0.846089,0.063095,13.409767,5.300490e-41,3.217773e-40,recip_syn2,18_aloifolia-V-18_filamentosa,YufilH1000007m.g,Yucal.01G000200.v2.1,18,Yf
1,67.211469,0.918770,0.064924,14.151414,1.830403e-45,1.182459e-44,recip_syn2,2AB_aloifolia-V-2AB_filamentosa,YufilH1000007m.g,Yucal.01G000200.v2.1,2AB,Yf
2,67.211469,0.878624,0.070650,12.436335,1.659451e-35,1.027305e-34,recip_syn2,1AB_aloifolia-V-1AB_filamentosa,YufilH1000007m.g,Yucal.01G000200.v2.1,1AB,Yf
3,67.211469,1.056236,0.085789,12.311974,7.808925e-35,7.650846e-34,recip_syn2,19_aloifolia-V-19_filamentosa,YufilH1000007m.g,Yucal.01G000200.v2.1,19,Yf
4,67.211469,0.779668,0.146913,5.307008,1.114394e-07,5.424920e-07,recip_syn2,15_aloifolia-V-15_filamentosa,YufilH1000007m.g,Yucal.01G000200.v2.1,15,Yf


In [5]:
# load HybridExpress results dataframe
res = pd.read_csv("./expression_partitioning_downstream/hybexp_partitiongenes_bygttreat.txt",sep="\t",header="infer")
res.head()

,Gene,Category,Class,lFC_F1_vs_P1,lFC_F1_vs_P2,genotype_treat
0,recip_syn1000,1,ADD,6.110441,-0.780159,18_D
1,recip_syn10003,1,ADD,3.313729,-0.887941,18_D
2,recip_syn10008,1,ADD,1.461161,-0.976261,18_D
3,recip_syn10061,1,ADD,2.826917,-1.349656,18_D
4,recip_syn10063,1,ADD,1.728989,-2.551956,18_D


In [6]:
# split genotype & treatment info
gtt = res["genotype_treat"].str.split("_",expand=True)
gtt.rename(columns={0:"genotype",1:"treat"},inplace=True)
res = pd.concat([res,gtt],axis=1)
res.head()

,Gene,Category,Class,lFC_F1_vs_P1,lFC_F1_vs_P2,genotype_treat,genotype,treat
0,recip_syn1000,1,ADD,6.110441,-0.780159,18_D,18,D
1,recip_syn10003,1,ADD,3.313729,-0.887941,18_D,18,D
2,recip_syn10008,1,ADD,1.461161,-0.976261,18_D,18,D
3,recip_syn10061,1,ADD,2.826917,-1.349656,18_D,18,D
4,recip_syn10063,1,ADD,1.728989,-2.551956,18_D,18,D


In [10]:
# for HybridExpress results: create a dict where key=genotype, value=syntelog ID list (all syntelogs DE between parents & each
## genotype)
gt_par_degs = {}
for g in res["genotype"].unique():
    df = res[res["genotype"]==g]
    gt_par_degs[g] = list(df["Gene"].unique())

In [12]:
# now make another dict: key=genotype, value = list of genes with HEB & no parental DE (cis only)
cisonly = {}
for g in heb["genotype"].unique():
    df = heb[(heb["genotype"]==g) & (~heb["syntelogID"].isin(gt_par_degs[g]))]
    cisonly[g] = list(df["syntelogID"].unique())

In [13]:
for k,v in cisonly.items():
    print("Genotype: "+k+", number of cis-only genes: "+str(len(v)))

Genotype: 18, number of cis-only genes: 1778
Genotype: 2AB, number of cis-only genes: 1604
Genotype: 1AB, number of cis-only genes: 1676
Genotype: 19, number of cis-only genes: 1927
Genotype: 15, number of cis-only genes: 2666
Genotype: Eudy, number of cis-only genes: 2699
Genotype: G, number of cis-only genes: 1873
Genotype: 56, number of cis-only genes: 1907
Genotype: 36, number of cis-only genes: 1961
Genotype: 13, number of cis-only genes: 2509
Genotype: 45, number of cis-only genes: 2164
Genotype: 52, number of cis-only genes: 1985
Genotype: 43, number of cis-only genes: 2186
Genotype: 37, number of cis-only genes: 2025
Genotype: 55, number of cis-only genes: 1826
Genotype: 70, number of cis-only genes: 2209
Genotype: 61, number of cis-only genes: 2127
Genotype: 51, number of cis-only genes: 2026
Genotype: 46, number of cis-only genes: 2384
Genotype: 53, number of cis-only genes: 2478
Genotype: 48, number of cis-only genes: 2563


In [14]:
# save cisonly
with open("./cisonly_posthybexp_25-02-2026.json","w+") as outfile:
    json.dump(cisonly,outfile)

## Make bed files for HOMER

In [16]:
def gtf_to_bed(filepath):
    gtf = pd.read_csv(filepath,sep="\t",header=None,comment="#")
    genes = gtf[gtf[2]=="gene"]
    
    gids = []
    for i in list(genes[8]):
        gids.append(i.strip().split(";")[0].split("=")[1])
    genes[9] = gids
    
    # create columns with start & stop of 1kb promoter
    ## for reverse-stranded genes: promoter will start from the endpoint (column 4) - i.e. will be to the 'right' of the gene
    pstart = []
    pstop = []
    for i in range(len(genes.index)):
        strand = genes.iloc[i,6]
        if strand=="+":
            refpt = genes.iloc[i,3]-1
            pstart.append(refpt-1001)
            pstop.append(refpt-1)
        elif strand=="-":
            refpt = genes.iloc[i,4]+1
            pstart.append(refpt+1)
            pstop.append(refpt+1001)
            
    genes[10] = pstart
    genes[11] = pstop
    
    bed = genes[[0,10,11,9,2,6]]
    return bed

In [17]:
yabed = gtf_to_bed("/home/leviathan22/Yucca_genomics/Yaloifolia_all.gtf_corrected.gtf")

/tmp/ipykernel_233/2335614318.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  genes[9] = gids
/tmp/ipykernel_233/2335614318.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  genes[10] = pstart
/tmp/ipykernel_233/2335614318.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-v

In [18]:
yfbed = gtf_to_bed("/home/leviathan22/Yucca_genomics/YfilamentosavarC3HAP1v3.1.gene.gtf_corrected.gtf")

/tmp/ipykernel_233/2335614318.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  genes[9] = gids
/tmp/ipykernel_233/2335614318.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  genes[10] = pstart
/tmp/ipykernel_233/2335614318.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-v

In [20]:
yabed.head()

,0,10,11,9,2,6
0,Chr01,30807,31807,Yucal.01G000100.v2.1,gene,-
12,Chr01,82673,83673,Yucal.01G000200.v2.1,gene,-
57,Chr01,51110,52110,Yucal.01G000300.v2.1,gene,-
61,Chr01,151105,152105,Yucal.01G000400.v2.1,gene,+
182,Chr01,191562,192562,Yucal.01G000500.v2.1,gene,+


In [21]:
# load long reciprocal syntelog data
syn = pd.read_csv("/home/leviathan22/Yucca_genomics/yucca_synteny/reciprocal_syntelogs_sameinYaYf_long.txt",sep="\t",header="infer")
syn.head()

,GeneID,syntelogID
0,YufilH1000002m.g,recip_syn1
1,Yucal.01G000100.v2.1,recip_syn1
2,YufilH1000007m.g,recip_syn2
3,Yucal.01G000200.v2.1,recip_syn2
4,YufilH1000011m.g,recip_syn3


In [22]:
# make a function to output (and write) a bed file for a given gene list (as referenced in the 'genelists' dict)
def write_gtbias_bed(gt,bias,topdir,bed):
    synlist = cisonly[gt]
    syndf = syn[syn["syntelogID"].isin(synlist)]
    glist = []
    if bias=="Ya":
        for i in syndf["GeneID"].unique():
            if i.startswith("Yucal"):
                glist.append(i)
    else:
        for i in syndf["GeneID"].unique():
            if i.startswith("Yufil"):
                glist.append(i)
    
    subbed = bed[bed[9].isin(glist)]
    
    # write subset bed file
    subbed.to_csv(os.path.join(topdir,gt+"_"+bias+"_promoters_forHOMER.bed"),sep="\t",header=False,index=False)

In [26]:
direc = "/home/leviathan22/yucca-genomics/cisgenes_promoters_2/"
if not os.path.exists(direc):
    os.makedirs(direc)
for k in cisonly.keys():
    for i,j in {"Ya":yabed,"Yf":yfbed}.items():
        write_gtbias_bed(k,i,direc,j)